张量类似Numpy的ndarray;

In [ ]:
import torch
import numpy as np

In [ ]:
#张量的初始化

#从数据创建
data=[[1,2],[3,4]]
x_data=torch.tensor(data)

#从NumpyPy创建
np_array=np.array(data)
print(type(np_array))
x_np=torch.from_numpy(np_array)
print(type(x_np))

#从另外的张量
x_ones=torch.ones_like(x_data)
print(x_ones)
x_rand=torch.rand_like(x_data,dtype=torch.float)
print(x_rand)

In [ ]:
#shape是张量维度的元组
shape=(2,3)
rand_tensor=torch.rand(shape)
ones_tensor=torch.ones(shape)
zeros_tensor=torch.zeros(shape)
print(rand_tensor)
print(ones_tensor)
print(zeros_tensor)

In [ ]:
#张量属性：形状，数据类型，存储他们的设备
tensor=torch.rand(3,4)
print(tensor.shape)
print(tensor.dtype)
print(tensor.device)

In [ ]:
#假设有GPU
if torch.cuda.is_avaiable():
    tensor=tensor.to('cuda')
    print(tensor.device)

In [ ]:
#标准的类似numpy的索引和切片
tensor=torch.ones(4,4)
print(tensor)
tensor[:,1]=0
print(tensor)

In [ ]:
#张量的链接
t1=torch.cat([tensor,tensor,tensor],dim=1)
print(t1)

In [ ]:
#乘以张量
print(tensor.mul(tensor))
#等价于
print('*'*30)
print(tensor*tensor)

In [ ]:
#张量的矩阵乘法
print(tensor.matmul(tensor.T))
print('*'*40)
print(tensor@tensor.T)

In [ ]:
#eg
x=[[1,2],[3,4]]
y=[[5,6],[7,8]]
x=torch.tensor(x)
y=torch.tensor(y)
print(x)
print(y)
print(type(x))
print(type(y))
#逐元素相乘
print(x*y)
print('*'*40)
print(x.mul(y))
print('*'*70)
#矩阵乘法
print(x@y)
print('*'*40)
print(x.matmul(y))

In [ ]:
#就地操作：带有后缀 _
print(tensor)
tensor.add_(5)
print(tensor)

In [ ]:
#cpu上的tensor 与 Numpy数组共享底部内存位置，更改其中一个会更改另外一个的
t=torch.ones(5)
print(t)
n=t.numpy()
print(n)

In [ ]:
t.add_(1)
print(t)
print(n)

In [ ]:
#Numpy数组转换为tensor
n=np.ones(5)
t=torch.from_numpy(n)
print(t)
print('*'*40)
np.add(n,1,out=n)#out表示输出位置 
print(t)
print(n)

In [ ]:
关于torch.autograd的介绍

In [6]:
#导入 PyTorch 和 torchvision 中的 ResNet-18 模型及预训练权重
import torch
from torchvision.models import resnet18,ResNet18_Weights

model=resnet18(weights=ResNet18_Weights.DEFAULT)
data=torch.rand(1,3,64,64)
'''
创建随机输入数据​​：
1：批次大小（batch size），表示一次处理1个样本
3：通道数，RGB三通道
64, 64：图像高度和宽度（64×64像素）
这是一个形状为 [1, 3, 64, 64]的随机张量
'''
labels=torch.rand(1,1000)
'''
创建随机标签​​：
1：批次大小，对应1个样本
1000：输出维度，对应1000个类别的概率分布
这是一个形状为 [1,  1000]的随机张量
'''

'\n创建随机标签\u200b\u200b：\n1：批次大小，对应1个样本\n1000：输出维度，对应1000个类别的概率分布\n这是一个形状为 [1,  1000]的随机张量\n'

In [7]:
#将输入的数据通过模型的每一层进行预测，这就是向前传播
prediction=model(data)

In [8]:
loss=(prediction-labels).sum()
#开始反向传播，梯度存储在.grad中
loss.backward()

In [9]:
#加载优化器，学习率为0.01，动量为0.9的SGD
optim=torch.optim.SGD(model.parameters(),lr=1e-2,momentum=0.9)

In [10]:
#调用.step()来启动梯度下降
optim.step()

In [1]:
import torch
#requires_grad=True 表示autograd对他们的操作被记录
a=torch.tensor([2.,3.],requires_grad=True)
b=torch.tensor([6.,4.],requires_grad=True)

In [2]:
Q=3*a**3-b**2

In [3]:
Q_sum=Q.sum()
Q_sum.backward()
print(a.grad)
print(b.grad)

tensor([36., 81.])
tensor([-12.,  -8.])


In [ ]:
external_grad=torch.tensor([1.,1.])
Q.backward(gradient=external_grad)

print(9*a**2==a.grad)
print(-2*b==b.grad)

In [4]:
x=torch.rand(5,5)
y=torch.rand(5,5)
z=torch.rand((5,5),requires_grad=True)
a=x+y
b=x+z
print(a.requires_grad)
print(b.requires_grad)

False
True


In [11]:
from torch import nn,optim
model=resnet18(weights=ResNet18_Weights.DEFAULT)
for param in model.parameters():
    param.requires_grad=False

In [13]:
model.fc=nn.Linear(512,10)
optimizer = optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)

In [ ]:
神经网络

In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class Net(nn.Module):
    def __init__(self):
        super(Net,self).__init__()
        self.conv1=nn.Conv2d(1,6,5)
        self.conv2=nn.Conv2d(6,16,5)
        self.fc1=nn.Linear(16*5*5,120)
        self.fc2=nn.Linear(120,84)
        self.fc3=nn.Linear(84,10)

    def forward(self,input):
        #卷积层
        c1=F.relu(self.conv1(input))
        #池化层1
        s2=F.max_pool2d(c1,(2,2))
        c3=F.relu(self.conv2(s2))
        s4=F.max_pool2d(c3,2)
        s4=torch.flatten(s4,1)
        f5=F.relu(self.fc1(s4))
        f6=F.relu(self.fc2(f5))
        output=self.fc3(f6)
        return output

net=Net()
print(net)
        
        

Net(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)


In [28]:
params=list(net.parameters())
print(len(params))
print(params[0].size())

10
torch.Size([6, 1, 5, 5])


In [29]:
input=torch.randn(1,1,32,32)
out=net(input)
print(out)

tensor([[-0.0066, -0.0842, -0.0143,  0.0579, -0.0450, -0.0046,  0.0697, -0.0480,
          0.0532, -0.1228]], grad_fn=<AddmmBackward0>)


In [17]:
net.zero_grad()
out.backward(torch.randn(1,10))

In [18]:
output=net(input)
target=torch.randn(10)
print(target)
target=target.view(1,-1)
criterion=nn.MSELoss()
loss=criterion(output,target)
print(loss)

tensor([ 0.8460, -0.0116, -0.1603, -0.8589,  0.2634,  0.2951,  0.2271, -0.2257,
         1.3827,  0.0390])
tensor(0.3406, grad_fn=<MseLossBackward0>)


In [24]:
net.zero_grad()
print(net.conv1.bias.grad)
loss.backward()
print(net.conv1.bias.grad)

None
tensor([-0.0062, -0.0066, -0.0007,  0.0057,  0.0015,  0.0063])


In [25]:
learning_rate=0.01
for f in net.parameters():
    f.data.sub_(f.grad.data*learning_rate)

In [26]:
import torch.optim as optim
optimizer=optim.SGD(net.parameters(),lr=0.01)
optimizer.zero_grad()
output=net(input)
loss=criterion(output,target)
loss.backward()
optimizer.step()